# LeetCode 419 — Battleships in a Board bằng Python

Link đề: https://leetcode.com/problems/battleships-in-a-board/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Depth-First Search, Union-Find, Counting

Skeleton LeetCode:

```python
class Solution:
    def countBattleships(self, board: List[List[str]]) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một bảng `m x n` gồm ô tàu `'X'` và ô trống `'.'`; tàu chỉ nằm **ngang** (`1 x k`) hoặc **dọc** (`k x 1`) và **không có hai tàu dính nhau** theo chiều ngang/dọc; hãy đếm số tàu.
2. Nắm công cụ Python: danh sách hai chiều `board[i][j]`, hai vòng `for` lồng nhau, `tuple` hướng đi `(di, dj)`, `set` các ô đã thăm, ngăn xếp (`list`) thay đệ quy, và cái bẫy **chỉ số âm** `board[-1]`.
3. Đi từ **tô loang (flood fill/DFS) đếm thành phần liên thông** → **Union-Find** → **đếm `đỉnh - cạnh` của rừng** → **chỉ đếm ô "góc trên-trái" của mỗi tàu** (một lượt, `O(1)` bộ nhớ, không sửa bảng — đúng yêu cầu follow-up).
4. Hiểu mấu chốt: vì tàu là **đoạn thẳng** và **không dính nhau**, mỗi tàu có đúng một ô mà phía trên và bên trái đều không phải `'X'`. Đếm các ô đó là đếm tàu.
5. Tránh các lỗi kinh điển: dùng `board[i-1][j]` khi `i = 0` (Python quay vòng sang hàng cuối), quên đánh dấu đã thăm gây lặp vô hạn, lẫn lộn `'X'` với `"X"`/`'x'`, và dùng thuật toán góc trên-trái cho bảng **không hợp lệ**.

## 2. Đề bài & ví dụ

Cho ma trận `board` kích thước `m x n`. Mỗi ô là phần của một con tàu `'X'` hoặc ô trống `'.'`. Trả về **số tàu chiến** (battleship) trên `board`.

Tàu chỉ có thể đặt **nằm ngang** hoặc **thẳng đứng**: hình dạng `1 x k` (một hàng, `k` cột) hoặc `k x 1` (`k` hàng, một cột), `k` tùy ý. **Giữa hai tàu luôn có ít nhất một ô trống theo chiều ngang hoặc dọc** (không có hai tàu liền kề nhau). Ghi chú: hai tàu vẫn có thể chạm nhau **chéo** ở góc.

**Ví dụ 1:**

```
X . . X
. . . X
. . . X
```
`board = [["X",".",".","X"],[".",".",".","X"],[".",".",".","X"]]` → `2` (một tàu `1x1` ở góc trái trên và một tàu `3x1` ở cột cuối).

**Ví dụ 2:** `board = [["."]]` → `0`.

**Ràng buộc:** `m == board.length`, `n == board[i].length`, `1 <= m, n <= 200`, `board[i][j]` là `'.'` hoặc `'X'`.

**Follow up:** làm được trong **một lượt duyệt**, chỉ dùng `O(1)` bộ nhớ phụ và **không sửa** giá trị của `board` không?

In [1]:
from typing import List
import random
import time

examples = [
    ([["X", ".", ".", "X"], [".", ".", ".", "X"], [".", ".", ".", "X"]], 2),
    ([["."]], 0),
]
for board, expected in examples:
    print("\n".join(" ".join(row) for row in board))
    print("-> kỳ vọng", expected)
    print()

X . . X
. . . X
. . . X
-> kỳ vọng 2

.
-> kỳ vọng 0



### 2.1. Đọc kỹ đề: điều kiện nào cho ta "đi đường tắt"?

Điều quan trọng nhất trong đề là **hai bảo đảm** về dữ liệu:

1. Mỗi tàu là **đoạn thẳng** (không có chữ L, chữ T, dấu cộng, hình vuông `2x2`...).
2. Hai tàu **không dính nhau theo 4 hướng** (trên, dưới, trái, phải). Chạm chéo thì được.

Nếu không có hai bảo đảm này, bài trở thành "đếm số vùng liên thông các `'X'`" tổng quát (cần flood fill). Có chúng, ta có thể dùng những mẹo nhẹ hơn nhiều. Nhưng khi dùng mẹo, **phải nhớ rằng nó chỉ đúng cho bảng hợp lệ**.

In [2]:
# Hai tàu chạm CHÉO là hợp lệ: (0,0) và (1,1) không liền kề theo 4 hướng
board = [["X", "."],
         [".", "X"]]
print("hợp lệ, 2 tàu 1x1 chạm chéo")

# Hai ô X cạnh nhau -> chúng là CÙNG một tàu 1x2, không phải 2 tàu
board = [["X", "X"]]
print("1 tàu 1x2")

hợp lệ, 2 tàu 1x1 chạm chéo
1 tàu 1x2


## 3. Kiến thức nền

### 3.1. Danh sách hai chiều (ma trận): `board[i][j]`

`board` là **danh sách các hàng**; `board[i]` là hàng thứ `i`; `board[i][j]` là ô ở hàng `i`, cột `j` (đếm từ `0`). `len(board)` là số hàng `m`, `len(board[0])` là số cột `n`.

In [3]:
board = [["X", ".", ".", "X"],
         [".", ".", ".", "X"],
         [".", ".", ".", "X"]]
m, n = len(board), len(board[0])
print("m =", m, "n =", n)
print("board[0] =", board[0])
print("board[1][3] =", board[1][3])

m = 3 n = 4
board[0] = ['X', '.', '.', 'X']
board[1][3] = X


### 3.2. Hai vòng `for` lồng nhau để duyệt mọi ô

`for i in range(m): for j in range(n):` thăm lần lượt từng ô theo thứ tự **từng hàng, trái sang phải** — thứ tự này sẽ rất quan trọng ở Cách 4 (khi ta đến ô `(i, j)` thì ô trên và ô trái đã được thăm).

In [4]:
order = []
for i in range(m):
    for j in range(n):
        order.append((i, j))
print(order[:6], "...", order[-2:])
print("tổng số ô:", len(order), "= m * n =", m * n)

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1)] ... [(2, 2), (2, 3)]
tổng số ô: 12 = m * n = 12


### 3.3. Cái bẫy chỉ số âm: `board[-1]` KHÔNG báo lỗi

Trong Python, chỉ số âm đếm từ cuối: `board[-1]` là **hàng cuối**. Vì thế `board[i - 1][j]` khi `i = 0` không lỗi mà lặng lẽ trả về ô ở **hàng cuối** — một kết quả sai âm thầm! Còn chỉ số quá lớn (`board[m]`) mới ném `IndexError`. Vì vậy ta phải **tự kiểm tra biên** (`i > 0`, `j > 0`...).

In [5]:
print("board[-1] =", board[-1], "(hàng cuối)")
print("board[0 - 1][3] =", board[0 - 1][3], "(thực ra là ô (2,3), không phải 'ngoài bảng')")
try:
    print(board[3])
except IndexError as e:
    print("IndexError:", e)

board[-1] = ['.', '.', '.', 'X'] (hàng cuối)
board[0 - 1][3] = X (thực ra là ô (2,3), không phải 'ngoài bảng')
IndexError: list index out of range


### 3.4. Hướng đi: danh sách `tuple` `(di, dj)`

Để đi sang 4 ô láng giềng ta dùng `dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]` (lên, xuống, trái, phải). Ô láng giềng của `(i, j)` là `(i + di, j + dj)`. Cần kiểm tra `0 <= ni < m and 0 <= nj < n`.

In [6]:
dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
i, j = 1, 3
for di, dj in dirs:
    ni, nj = i + di, j + dj
    inside = 0 <= ni < m and 0 <= nj < n
    print((ni, nj), "trong bảng" if inside else "NGOÀI bảng", board[ni][nj] if inside else "")

(0, 3) trong bảng X
(2, 3) trong bảng X
(1, 2) trong bảng .
(1, 4) NGOÀI bảng 


### 3.5. `set` các ô đã thăm và `stack` (ngăn xếp) bằng `list`

- `visited = set()`; `visited.add((i, j))`; `(i, j) in visited` mất `O(1)` trung bình. `tuple` băm được nên đưa vào `set` thoải mái; `list` thì không.
- `stack = [start]`; `stack.append(x)` đẩy vào; `stack.pop()` lấy phần tử **cuối**. Thay đệ quy bằng ngăn xếp tránh lỗi tràn đệ quy.

In [7]:
visited = set()
visited.add((0, 0))
print((0, 0) in visited, (0, 1) in visited)

stack = [(0, 0)]
stack.append((1, 1))
print(stack.pop(), stack)       # (1, 1) ra trước (vào sau ra trước - LIFO)

True False
(1, 1) [(0, 0)]


### 3.6. `all`, `any`, `sum` với generator và đếm ô `'X'`

`sum(row.count('X') for row in board)` đếm tổng số ô `'X'`. Ta sẽ dùng ở Cách 3.

In [8]:
print(sum(row.count("X") for row in board))     # 4
print(any("X" in row for row in board), all("X" in row for row in board))

4
True True


## 4. Ý tưởng nền tảng: tính tay một ví dụ

Lấy ví dụ 1. Ta nhìn bảng và đếm "bằng mắt": có `X` đơn lẻ ở góc trái trên (1 tàu), và 3 ô `X` xếp dọc ở cột 3 (1 tàu). Tổng **2 tàu** trong khi có **4 ô `X`**.

Câu hỏi: làm sao để máy đếm được mà không cần "nhìn"? Hai hướng nghĩ tự nhiên:

- **Hướng 1 — theo thành phần:** bắt đầu từ một ô `X` chưa thăm, "tô loang" ra mọi ô `X` liền kề để đánh dấu cả con tàu, rồi đếm 1. Lặp cho tới khi hết ô `X` chưa thăm.
- **Hướng 2 — theo đại diện:** mỗi tàu chọn ra **một ô đặc trưng** (ví dụ ô đầu tiên khi quét từ trái sang phải, từ trên xuống dưới) và đếm các ô đại diện đó.

Hướng 2 đòi hỏi phải trả lời: ô đầu tiên của một tàu được nhận ra thế nào? Quét từ trên xuống, trái sang phải thì ô đầu tiên gặp phải của một tàu có **phía trên và bên trái không phải `X`**. Hãy thử với ví dụ 1.

In [9]:
def is_top_left(board, i, j):
    if board[i][j] != "X":
        return False
    up_is_x = i > 0 and board[i - 1][j] == "X"
    left_is_x = j > 0 and board[i][j - 1] == "X"
    return not up_is_x and not left_is_x


board = examples[0][0]
for i in range(len(board)):
    for j in range(len(board[0])):
        if board[i][j] == "X":
            print((i, j), "góc trên-trái?", is_top_left(board, i, j))

(0, 0) góc trên-trái? True
(0, 3) góc trên-trái? True
(1, 3) góc trên-trái? False
(2, 3) góc trên-trái? False


Bốn ô `X`: `(0,0)` là góc; `(0,3)` là góc (phía trên là biên, bên trái là `.`); `(1,3)` có `X` ở trên ⇒ không phải; `(2,3)` có `X` ở trên ⇒ không phải. Có đúng **2 ô góc** ⇒ 2 tàu. Ta sẽ chứng minh vì sao điều này luôn đúng ở mục 10.

## 5. Cách 1 — Flood fill (DFS) với tập đã thăm

**Ý tưởng (đếm thành phần liên thông):**

1. Duyệt mọi ô `(i, j)`. Nếu là `'X'` và **chưa thăm** ⇒ tìm thấy một tàu mới: `ans += 1`.
2. Từ ô đó chạy DFS: thăm mọi `'X'` liền kề theo 4 hướng, đánh dấu vào `visited` để không đếm lại.

Ta dùng **ngăn xếp** (DFS lặp) cho an toàn, và `visited` là tập riêng để **không sửa** `board`.

In [10]:
def count_dfs(board):
    m, n = len(board), len(board[0])
    visited = set()
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X" and (i, j) not in visited:
                ans += 1                              # gặp một tàu mới
                visited.add((i, j))
                stack = [(i, j)]
                while stack:                          # tô loang cả con tàu
                    x, y = stack.pop()
                    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "X" and (nx, ny) not in visited:
                            visited.add((nx, ny))
                            stack.append((nx, ny))
    return ans


for board, expected in examples:
    got = count_dfs(board)
    print(got, "| kỳ vọng", expected)
    assert got == expected

2 | kỳ vọng 2
0 | kỳ vọng 0


**Độ phức tạp:** mỗi ô vào ngăn xếp tối đa một lần, mỗi lần xét 4 láng giềng ⇒ **Time `O(m * n)`**; `visited` + ngăn xếp tối đa `O(m * n)` ⇒ **Space `O(m * n)`**.

**Nút thắt:** đúng nhưng **tốn bộ nhớ** và **không tận dụng** việc tàu là đoạn thẳng, không dính nhau. Follow-up yêu cầu `O(1)` bộ nhớ, một lượt, không sửa bảng. Ta cần nhìn ra cấu trúc đặc biệt của bài.

## 6. Ý tưởng thuật toán / bước đột phá

Vì mỗi tàu là **đoạn thẳng** và các tàu **không dính nhau**, mỗi tàu có một **ô đầu tiên** duy nhất khi quét theo thứ tự đọc (trên → dưới, trái → phải):

- Tàu ngang `1 x k`: ô đầu tiên là ô **trái nhất**. Bên trái nó là `'.'` hoặc biên (nếu có `X` ở bên trái thì ô đó thuộc **cùng** tàu, mâu thuẫn với "ô trái nhất"; hoặc thuộc tàu khác dính kề, bị cấm). Phía trên nó là `'.'` hoặc biên (nếu có `X` ở trên thì tàu sẽ dính với tàu khác, hoặc tàu không còn là đoạn thẳng).
- Tàu dọc `k x 1`: ô đầu tiên là ô **trên cùng**; phía trên nó là `'.'`/biên và bên trái cũng vậy (cùng lý do).
- Mọi ô khác của tàu đều có **ô trước** nó (bên trái với tàu ngang, phía trên với tàu dọc) là `'X'` ⇒ không phải ô góc.

⇒ **Số tàu = số ô `'X'` mà ô phía trên và ô bên trái đều không phải `'X'`.**

Để chắc chắn, hãy kiểm chứng ý này bằng số trên ví dụ đề rồi sẽ so với flood fill trên nhiều bảng ngẫu nhiên hợp lệ ở mục 12.

In [11]:
for board, expected in examples:
    corners = sum(
        1
        for i in range(len(board))
        for j in range(len(board[0]))
        if board[i][j] == "X" and (i == 0 or board[i - 1][j] != "X") and (j == 0 or board[i][j - 1] != "X")
    )
    print("số ô góc =", corners, "| kỳ vọng", expected)
    assert corners == expected

số ô góc = 2 | kỳ vọng 2
số ô góc = 0 | kỳ vọng 0


## 7. Các cách tiếp theo

### Cách 2 — Union-Find (disjoint set union): gộp các ô `X` liền kề

**Giải thích từng đoạn:**

1. Mỗi ô `(i, j)` được đánh số `id = i * n + j`. `parent[id]` ban đầu là chính nó.
2. Với mỗi ô `X`, nếu ô **phía trên** là `X` thì `union(id, id_trên)`; nếu ô **bên trái** là `X` thì `union(id, id_trái)`. (Chỉ cần hai hướng vì mỗi cặp liền kề được xét đúng một lần khi quét.)
3. Số tàu = số **gốc** (root) khác nhau trong các ô `X`; hoặc đếm `X` rồi trừ một cho mỗi lần `union` thực sự gộp được hai tập khác nhau.

`find(x)` dùng **nén đường đi** (path compression) để gần như `O(1)`.

In [12]:
def count_union_find(board):
    m, n = len(board), len(board[0])
    parent = list(range(m * n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]       # nén đường đi: nhảy cóc lên ông bà
            x = parent[x]
        return x

    components = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] != "X":
                continue
            components += 1                      # tạm coi là tàu mới
            for ni, nj in ((i - 1, j), (i, j - 1)):
                if ni >= 0 and nj >= 0 and board[ni][nj] == "X":
                    a, b = find(i * n + j), find(ni * n + nj)
                    if a != b:
                        parent[a] = b
                        components -= 1          # gộp hai tập -> bớt một thành phần
    return components


for board, expected in examples:
    assert count_union_find(board) == expected
print("Cách 2 đúng trên ví dụ đề")

Cách 2 đúng trên ví dụ đề


In [13]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def countBattleships(self, board: List[List[str]]) -> int:
        m, n = len(board), len(board[0])
        parent = list(range(m * n))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        ans = 0
        for i in range(m):
            for j in range(n):
                if board[i][j] != "X":
                    continue
                ans += 1
                for ni, nj in ((i - 1, j), (i, j - 1)):
                    if ni >= 0 and nj >= 0 and board[ni][nj] == "X":
                        a, b = find(i * n + j), find(ni * n + nj)
                        if a != b:
                            parent[a] = b
                            ans -= 1
        return ans


print(Solution2().countBattleships(examples[0][0]))   # 2

2


**Độ phức tạp:** `O(m * n * α(m * n))` với `α` là hàm Ackermann ngược (coi như hằng số) ⇒ ~`O(m * n)`; Space `O(m * n)` cho `parent`.

**Nhận xét:** Union-Find tổng quát hơn (đúng cả khi tàu không phải đoạn thẳng), cũng đúng với mọi bảng `X/.` bất kỳ, nhưng không giảm bộ nhớ.

### Cách 3 — Đếm `đỉnh - cạnh` của một rừng (Euler)

**Ý tưởng toán học:** coi mỗi ô `X` là một **đỉnh**, hai ô `X` kề nhau (trên-dưới hoặc trái-phải) là một **cạnh**. Mỗi tàu là một **đoạn thẳng** ⇒ đồ thị con tương ứng là một **đường đi** (cây). Một cây có `V` đỉnh có đúng `V - 1` cạnh. Nên với một **rừng** (nhiều cây rời nhau):

`số cây = V - E`.

Vậy số tàu = `(số ô X) - (số cặp ô X kề nhau)`. Đếm cặp kề nhau chỉ cần xét **hướng lên** và **hướng sang trái** (mỗi cặp đúng một lần).

**Giải thích từng đoạn:**

1. `V` = tổng số ô `X`.
2. `E` = số `(i, j)` có `X` và ô trên là `X`, cộng số `(i, j)` có `X` và ô trái là `X`.
3. Trả `V - E`.

Chú ý: công thức này chỉ đúng nếu không có **chu trình** (không có khối `2x2` toàn `X`). Bảng hợp lệ của đề đảm bảo điều đó.

In [14]:
def count_forest(board):
    m, n = len(board), len(board[0])
    V = E = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] != "X":
                continue
            V += 1
            if i > 0 and board[i - 1][j] == "X":
                E += 1
            if j > 0 and board[i][j - 1] == "X":
                E += 1
    return V - E


for board, expected in examples:
    got = count_forest(board)
    print("V - E =", got, "| kỳ vọng", expected)
    assert got == expected

V - E = 2 | kỳ vọng 2
V - E = 0 | kỳ vọng 0


In [15]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def countBattleships(self, board: List[List[str]]) -> int:
        m, n = len(board), len(board[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if board[i][j] == "X":
                    ans += 1                                        # +1 cho mỗi đỉnh
                    if i > 0 and board[i - 1][j] == "X":
                        ans -= 1                                    # -1 cho mỗi cạnh lên
                    if j > 0 and board[i][j - 1] == "X":
                        ans -= 1                                    # -1 cho mỗi cạnh trái
        return ans


print(Solution3().countBattleships([["X", "X", "."], [".", ".", "X"]]))   # tàu 1x2 + tàu 1x1 = 2

2


**Độ phức tạp:** Time `O(m * n)`, Space `O(1)`. Cùng điều kiện đúng như Cách 4: cần bảng hợp lệ (không chu trình).

### Cách 4 — Đếm ô góc trên-trái (câu trả lời cho follow-up)

**Giải thích từng đoạn:**

1. Duyệt mọi ô theo thứ tự đọc.
2. Bỏ qua ô `'.'`.
3. Nếu ô trên (khi `i > 0`) là `'X'` ⇒ ô này thuộc phần thân của tàu dọc ⇒ bỏ qua.
4. Nếu ô trái (khi `j > 0`) là `'X'` ⇒ ô này thuộc phần thân của tàu ngang ⇒ bỏ qua.
5. Ngược lại ⇒ ô góc ⇒ `ans += 1`.

Không dùng bộ nhớ phụ, không sửa bảng, đúng **một lượt**.

In [16]:
def count_corner(board):
    m, n = len(board), len(board[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == ".":
                continue
            if i > 0 and board[i - 1][j] == "X":
                continue
            if j > 0 and board[i][j - 1] == "X":
                continue
            ans += 1
    return ans


for board, expected in examples:
    got = count_corner(board)
    print(got, "| kỳ vọng", expected)
    assert got == expected

2 | kỳ vọng 2
0 | kỳ vọng 0


In [17]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def countBattleships(self, board: List[List[str]]) -> int:
        m, n = len(board), len(board[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if board[i][j] == ".":
                    continue
                if i > 0 and board[i - 1][j] == "X":
                    continue
                if j > 0 and board[i][j - 1] == "X":
                    continue
                ans += 1
        return ans


print(Solution4().countBattleships([["X", ".", "X"], ["X", ".", "X"]]))   # 2 tàu dọc

2


**Độ phức tạp:** Time `O(m * n)`, Space `O(1)`.

**Bốn cách tóm tắt:** (1) tô loang DFS; (2) Union-Find gộp ô kề; (3) rừng: `V - E`; (4) đếm ô góc. Ba cách đầu thể hiện ba lăng kính khác nhau của "đếm thành phần liên thông"; cách 4 dùng thêm tính chất đặc biệt của đề để đạt `O(1)` bộ nhớ.

## 8. TRACE từng bước

Ta trace trên một bảng `4 x 5` có nhiều dạng tàu:

```
X X X . X
. . . . X
X . . . .
X . X . X
```

Các tàu: ngang `1x3` (hàng 0, cột 0-2), dọc `2x1` (cột 4, hàng 0-1), dọc `2x1` (cột 0, hàng 2-3), đơn `(3,2)`, đơn `(3,4)`. Tổng **5 tàu**.

In [18]:
trace_board = [list("XXX.X"),
               list("....X"),
               list("X...."),
               list("X.X.X")]
print(count_dfs(trace_board), count_union_find(trace_board), count_forest(trace_board), count_corner(trace_board))

5 5 5 5


Bảng trạng thái của **Cách 4** (mỗi dòng là một ô `X`):

| ô | ô trên | ô trái | kết luận | `ans` |
|---|---|---|---|---|
| (0,0) | biên | biên | góc | 1 |
| (0,1) | biên | `X` | thân | 1 |
| (0,2) | biên | `X` | thân | 1 |
| (0,4) | biên | `.` | góc | 2 |
| (1,4) | `X` | `.` | thân | 2 |
| (2,0) | `.` | biên | góc | 3 |
| (3,0) | `X` | biên | thân | 3 |
| (3,2) | `.` | `.` | góc | 4 |
| (3,4) | `.` | `.` | góc | 5 |

Hàm trace in đúng các dòng này.

In [19]:
def trace_corner(board):
    m, n = len(board), len(board[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == ".":
                continue
            up = "biên" if i == 0 else board[i - 1][j]
            left = "biên" if j == 0 else board[i][j - 1]
            if up == "X":
                verdict = "thân tàu dọc (trên là X)"
            elif left == "X":
                verdict = "thân tàu ngang (trái là X)"
            else:
                verdict = "GÓC -> tàu mới"
                ans += 1
            print(f"ô ({i},{j}): trên={up!s:>4}, trái={left!s:>4} -> {verdict:<28} ans={ans}")
    return ans


assert trace_corner(trace_board) == 5

ô (0,0): trên=biên, trái=biên -> GÓC -> tàu mới               ans=1
ô (0,1): trên=biên, trái=   X -> thân tàu ngang (trái là X)   ans=1
ô (0,2): trên=biên, trái=   X -> thân tàu ngang (trái là X)   ans=1
ô (0,4): trên=biên, trái=   . -> GÓC -> tàu mới               ans=2
ô (1,4): trên=   X, trái=   . -> thân tàu dọc (trên là X)     ans=2
ô (2,0): trên=   ., trái=biên -> GÓC -> tàu mới               ans=3
ô (3,0): trên=   X, trái=biên -> thân tàu dọc (trên là X)     ans=3
ô (3,2): trên=   ., trái=   . -> GÓC -> tàu mới               ans=4
ô (3,4): trên=   ., trái=   . -> GÓC -> tàu mới               ans=5


Bây giờ trace **Cách 3** (`V - E`) trên cùng bảng: mỗi ô `X` cộng 1 đỉnh, mỗi cạnh trừ 1.

In [20]:
def trace_forest(board):
    m, n = len(board), len(board[0])
    V = E = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] != "X":
                continue
            V += 1
            edges = []
            if i > 0 and board[i - 1][j] == "X":
                E += 1
                edges.append("lên")
            if j > 0 and board[i][j - 1] == "X":
                E += 1
                edges.append("trái")
            print(f"ô ({i},{j}): V={V}, cạnh mới: {edges or '-'}, E={E}, V-E={V - E}")
    return V - E


assert trace_forest(trace_board) == 5

ô (0,0): V=1, cạnh mới: -, E=0, V-E=1
ô (0,1): V=2, cạnh mới: ['trái'], E=1, V-E=1
ô (0,2): V=3, cạnh mới: ['trái'], E=2, V-E=1
ô (0,4): V=4, cạnh mới: -, E=2, V-E=2
ô (1,4): V=5, cạnh mới: ['lên'], E=3, V-E=2
ô (2,0): V=6, cạnh mới: -, E=3, V-E=3
ô (3,0): V=7, cạnh mới: ['lên'], E=4, V-E=3
ô (3,2): V=8, cạnh mới: -, E=4, V-E=4
ô (3,4): V=9, cạnh mới: -, E=4, V-E=5


Cuối cùng trace **Cách 1** (DFS) để thấy thứ tự thăm của một con tàu: ta in từng ô được lấy ra khỏi ngăn xếp.

In [21]:
def trace_dfs(board):
    m, n = len(board), len(board[0])
    visited = set()
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X" and (i, j) not in visited:
                ans += 1
                print(f"-- tàu #{ans} bắt đầu tại ({i},{j})")
                visited.add((i, j))
                stack = [(i, j)]
                while stack:
                    x, y = stack.pop()
                    print(f"   pop ({x},{y}); stack còn {stack}")
                    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "X" and (nx, ny) not in visited:
                            visited.add((nx, ny))
                            stack.append((nx, ny))
    return ans


assert trace_dfs(trace_board) == 5

-- tàu #1 bắt đầu tại (0,0)
   pop (0,0); stack còn []
   pop (0,1); stack còn []
   pop (0,2); stack còn []
-- tàu #2 bắt đầu tại (0,4)
   pop (0,4); stack còn []
   pop (1,4); stack còn []
-- tàu #3 bắt đầu tại (2,0)
   pop (2,0); stack còn []
   pop (3,0); stack còn []
-- tàu #4 bắt đầu tại (3,2)
   pop (3,2); stack còn []
-- tàu #5 bắt đầu tại (3,4)
   pop (3,4); stack còn []


## 9. Code nộp LeetCode

Bản khuyến nghị: **Cách 4** (đếm ô góc trên-trái) — ngắn, `O(m * n)` thời gian, `O(1)` bộ nhớ, không sửa `board`, một lượt duyệt, đúng follow-up.

In [22]:
class Solution:
    def countBattleships(self, board: List[List[str]]) -> int:
        m, n = len(board), len(board[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if board[i][j] == ".":
                    continue                      # ô trống
                if i > 0 and board[i - 1][j] == "X":
                    continue                      # có X phía trên: thân tàu dọc
                if j > 0 and board[i][j - 1] == "X":
                    continue                      # có X bên trái: thân tàu ngang
                ans += 1                          # góc trên-trái của một tàu mới
        return ans


for board, expected in examples:
    got = Solution().countBattleships(board)
    print(got, expected)
    assert got == expected

2 2
0 0


## 10. Vì sao đúng

**Khẳng định:** với bảng hợp lệ, mỗi tàu có **đúng một** ô `X` mà ô trên và ô trái đều không phải `X` (ta gọi là *ô góc*), và mọi ô góc đều thuộc đúng một tàu. Suy ra số ô góc = số tàu.

**Chứng minh.** Xét một tàu `T`.

- *Tàu ngang* gồm các ô `(r, c), (r, c+1), ..., (r, c+k-1)` (`k >= 1`). Ô `(r, c)`: bên trái `(r, c-1)` không thuộc `T`; nó phải là `'.'` hoặc biên vì nếu là `X` thì thuộc tàu khác **dính kề** `T` (cấm) — hoặc thuộc `T`, nhưng `T` bắt đầu từ `c`. Phía trên `(r-1, c)` cũng phải là `'.'`/biên vì nếu là `X` nó dính kề ô `(r, c)` của `T` ⇒ trái đề. Vậy `(r, c)` là ô góc. Các ô `(r, c + t)` với `t >= 1` có ô trái `(r, c + t - 1)` thuộc `T` ⇒ là `X` ⇒ **không** phải ô góc. Vậy `T` có đúng 1 ô góc.
- *Tàu dọc* tương tự, đổi vai trò trên/trái.
- *Tàu `1x1`* (cả ngang lẫn dọc): ô duy nhất có trên và trái đều không phải `X` (cùng lý do dính kề) ⇒ 1 ô góc.

Mọi ô `X` thuộc đúng một tàu, nên mọi ô góc cũng được tính cho đúng tàu của nó. Do đó `ans` = số tàu. ∎

**Vì sao bảng không hợp lệ phá vỡ lập luận?** Nếu hai tàu được phép dính nhau thì ô đầu của tàu thứ hai có thể có `X` ở trên (thuộc tàu thứ nhất) và bị bỏ sót; nếu tàu không phải đoạn thẳng (chữ L, chữ T) thì một tàu có thể có **hai** ô góc (ví dụ dấu cộng). Mục 13 minh họa cụ thể.

**Với Cách 3 (`V - E`):** trong bảng hợp lệ, mỗi tàu `1 x k` có `k` đỉnh và `k - 1` cạnh (các cặp kề nhau cùng hàng/cột); không có cạnh giữa hai tàu khác nhau (cấm dính kề) ⇒ `V - E = Σ(k_t - (k_t - 1)) = số tàu`.

**Với Cách 1 và 2:** số tàu = số thành phần liên thông của đồ thị lưới 4-hướng trên các ô `X` — đúng theo định nghĩa vì hai ô cùng tàu thì liên thông (đường đi dọc theo tàu) và hai ô khác tàu thì không (không có ô kề nhau).

## 11. Time / Space Complexity

| Cách | Time | Space | Sửa bảng? |
|---|---|---|---|
| 1. DFS + `visited` | `O(m * n)` | `O(m * n)` | Không |
| 2. Union-Find | `O(m * n * α)` | `O(m * n)` | Không |
| 3. `V - E` | `O(m * n)` | `O(1)` | Không |
| 4. Ô góc trên-trái | `O(m * n)` | `O(1)` | Không |

Với `m, n <= 200` thì `m * n = 4 * 10^4` ô: cả bốn cách chạy nhanh. Khác biệt chủ yếu là **bộ nhớ** và độ **tổng quát**. Hãy đo thời gian trên bảng lớn hơn ràng buộc (1000 x 1000) để thấy khác biệt tương đối.

In [23]:
def gen_board(m, n, attempts, rng):
    """Sinh bảng hợp lệ: đặt tàu ngẫu nhiên, mỗi tàu và 4-láng-giềng của nó phải trống. Trả (board, số tàu đã đặt, kích cỡ các tàu)."""
    grid = [["."] * n for _ in range(m)]
    sizes = []
    for _ in range(attempts):
        horizontal = rng.random() < 0.5
        k = rng.randint(1, max(1, (n if horizontal else m) // 2))
        if horizontal:
            if k > n:
                continue
            r, c = rng.randrange(m), rng.randrange(n - k + 1)
            cells = [(r, c + t) for t in range(k)]
        else:
            if k > m:
                continue
            r, c = rng.randrange(m - k + 1), rng.randrange(n)
            cells = [(r + t, c) for t in range(k)]
        ok = True
        for (x, y) in cells:
            for dx, dy in [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "X":
                    ok = False
        if ok:
            for (x, y) in cells:
                grid[x][y] = "X"
            sizes.append(k)
    return grid, len(sizes), sizes


rng = random.Random(419)
bd, cnt, sizes = gen_board(8, 12, 40, rng)
print("\n".join("".join(r) for r in bd))
print("số tàu đã đặt =", cnt, "| kích cỡ:", sizes)

X....X.X....
....X..X....
X...X...XXX.
X......X....
X..........X
.XXXXXX....X
.......XXX..
X...X.......
số tàu đã đặt = 12 | kích cỡ: [2, 2, 1, 3, 6, 1, 3, 1, 2, 1, 1, 3]


In [24]:
rng = random.Random(1)
big, truth, _ = gen_board(1000, 1000, 60000, rng)
print("bảng 1000 x 1000, số tàu thật =", truth)
for name, f in [("DFS", count_dfs), ("Union-Find", count_union_find), ("V - E", count_forest), ("Góc trên-trái", count_corner)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:<14}: {r} (đúng: {r == truth}) trong {time.perf_counter() - t:.3f} giây")

bảng 1000 x 1000, số tàu thật = 2544


DFS           : 2544 (đúng: True) trong 0.404 giây
Union-Find    : 2544 (đúng: True) trong 0.203 giây


V - E         : 2544 (đúng: True) trong 0.055 giây
Góc trên-trái : 2544 (đúng: True) trong 0.040 giây


## 12. Bộ test

Oracle độc lập: `gen_board` đặt tàu **bằng cách sinh**, nên số tàu thật là số lần đặt thành công — **không dùng bất kỳ thuật toán đếm nào**. Ta đối chiếu mọi cách với con số đó, và với DFS (đối chiếu chéo).

In [25]:
solutions = {
    "dfs": count_dfs,
    "union-find": count_union_find,
    "forest V-E": count_forest,
    "corner": count_corner,
    "LeetCode Solution": lambda b: Solution().countBattleships(b),
    "Solution2": lambda b: Solution2().countBattleships(b),
    "Solution3": lambda b: Solution3().countBattleships(b),
    "Solution4": lambda b: Solution4().countBattleships(b),
}

# 12.1. Ví dụ của đề
for board, expected in examples:
    for name, f in solutions.items():
        assert f(board) == expected, (name, board)
print("12.1 OK")

12.1 OK


In [26]:
# 12.2. Edge case
edge = [
    ([["X"]], 1),                                   # 1 ô, 1 tàu
    ([["."]], 0),
    ([["X", "X", "X", "X"]], 1),                    # một hàng, 1 tàu ngang
    ([["X"], ["X"], ["X"]], 1),                     # một cột, 1 tàu dọc
    ([["X", "."], [".", "X"]], 2),                  # chạm chéo vẫn là 2 tàu
    ([["X", ".", "X"], ["X", ".", "X"]], 2),        # hai tàu dọc song song cách 1 cột
    ([["X", ".", "X", ".", "X"]], 3),               # ba tàu đơn trên một hàng
    ([["."] * 5 for _ in range(3)], 0),             # toàn trống
    ([["X", "X", "."], [".", ".", "."], ["X", "X", "X"]], 2),
]
for board, expected in edge:
    for name, f in solutions.items():
        got = f(board)
        assert got == expected, (name, board, got, expected)
print("12.2 OK:", len(edge), "edge case x", len(solutions), "cách")

12.2 OK: 9 edge case x 8 cách


In [27]:
# 12.3. Ngẫu nhiên có seed: bảng hợp lệ, oracle = số tàu đã đặt
rng = random.Random(4190)
for _ in range(1500):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    board, truth, _ = gen_board(m, n, rng.randint(0, 30), rng)
    for name, f in solutions.items():
        got = f(board)
        assert got == truth, (name, board, got, truth)
print("12.3 OK: 1500 bảng hợp lệ ngẫu nhiên, mọi cách khớp số tàu thật")

12.3 OK: 1500 bảng hợp lệ ngẫu nhiên, mọi cách khớp số tàu thật


In [28]:
# 12.4. Bảng lớn đúng ràng buộc 200 x 200
rng = random.Random(77)
for _ in range(10):
    board, truth, _ = gen_board(200, 200, 3000, rng)
    assert Solution().countBattleships(board) == truth == count_dfs(board) == count_union_find(board) == count_forest(board)
print("12.4 OK: 10 bảng 200 x 200")

12.4 OK: 10 bảng 200 x 200


In [29]:
# 12.5. Kiểm tra Solution không sửa board (follow-up)
import copy
rng = random.Random(9)
board, truth, _ = gen_board(30, 30, 300, rng)
snapshot = copy.deepcopy(board)
Solution().countBattleships(board)
assert board == snapshot
print("12.5 OK: board không bị sửa")

12.5 OK: board không bị sửa


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `board[i - 1][j]` mà không kiểm tra `i > 0`

Ở `i = 0`, `board[-1]` là **hàng cuối**. Nếu hàng cuối có `X` ở cùng cột, ô ở hàng 0 bị coi nhầm là "thân tàu" và **bị bỏ sót**.

In [30]:
def wrong_negative_index(board):
    m, n = len(board), len(board[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X" and board[i - 1][j] != "X" and board[i][j - 1] != "X":   # SAI: không kiểm tra biên
                ans += 1
    return ans


bad = [["X", "."],
       ["X", "."],
       [".", "."],
       ["X", "."]]
# hàng 0 và hàng 3 đều có X ở cột 0; hàng cuối (3) "cuộn" lên làm ô (0,0) trông như có X phía trên
print("sai  :", wrong_negative_index(bad))
print("đúng :", count_corner(bad), "(tàu dọc 2x1 và tàu đơn ở hàng cuối)")

sai  : 1
đúng : 2 (tàu dọc 2x1 và tàu đơn ở hàng cuối)


**Vì sao sai?** `board[0 - 1]` là `board[3]` nên ô `(0, 0)` bị cho là có `X` phía trên ⇒ không tính góc ⇒ mất một tàu. Tương tự `board[i][j - 1]` ở `j = 0` sẽ nhìn sang **cột cuối**. Sửa: luôn `i > 0 and ...` và `j > 0 and ...`.

### Lỗi 2 — DFS quên đánh dấu đã thăm (đếm lặp / lặp vô hạn)

Nếu không đánh dấu `visited` thì mỗi ô `X` được đếm lại mỗi khi quét tới nó, và DFS đi qua lại giữa hai ô kề nhau mãi mãi. Ở dưới ta chỉ mô phỏng phiên bản quên đánh dấu **cho vòng ngoài** (vẫn đánh dấu trong DFS) để thấy việc đếm sai.

In [31]:
def wrong_no_outer_visited(board):
    m, n = len(board), len(board[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X":          # SAI: không kiểm tra (i, j) đã thuộc tàu nào chưa
                ans += 1
                seen = {(i, j)}
                stack = [(i, j)]
                while stack:
                    x, y = stack.pop()
                    for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "X" and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            stack.append((nx, ny))
    return ans


print("sai  :", wrong_no_outer_visited(examples[0][0]), "(đếm 4 = số ô X, thay vì 2 tàu)")
print("đúng :", count_dfs(examples[0][0]))

sai  : 4 (đếm 4 = số ô X, thay vì 2 tàu)
đúng : 2


**Vì sao sai?** Mỗi ô `X` được coi là một tàu mới vì `visited` không được dùng chung giữa các lần DFS. Sửa: tập `visited` phải **sống qua toàn bộ vòng ngoài**, và điều kiện bắt đầu DFS là `(i, j) not in visited`.

### Lỗi 3 — Áp thuật toán "ô góc" lên bảng **không hợp lệ**

Thuật toán ô góc chỉ đúng khi tàu là đoạn thẳng và không dính nhau. Với hình dấu cộng (một "tàu" không hợp lệ) nó cho 2.

In [32]:
plus = [list(".X."),
        list("XXX"),
        list(".X.")]
print("dấu cộng: ô góc =", count_corner(plus), "| thành phần liên thông (DFS) =", count_dfs(plus), "| V - E =", count_forest(plus))

block = [list("XX"),
         list("XX")]
print("khối 2x2: ô góc =", count_corner(block), "| DFS =", count_dfs(block), "| V - E =", count_forest(block), "(có chu trình!)")

dấu cộng: ô góc = 2 | thành phần liên thông (DFS) = 1 | V - E = 1
khối 2x2: ô góc = 1 | DFS = 1 | V - E = 0 (có chu trình!)


**Vì sao sai?** Dấu cộng có **hai** ô góc `(0,1)` và `(1,0)` ⇒ ô góc đếm 2, DFS đếm 1. Khối `2x2` có chu trình nên `V - E = 4 - 4 = 0`. Chỉ cần nhớ: mẹo chỉ đúng nhờ **bảo đảm của đề**. Nếu đề đổi thành "đếm vùng `X` liên thông bất kỳ" thì dùng DFS/Union-Find.

### Lỗi 4 — So sánh sai ký tự

Bảng dùng `'X'` (chữ **hoa**) và `'.'`. Viết `'x'` hoặc so sánh với `"X "` luôn sai và làm `ans = 0` hoặc đếm hết.

In [33]:
row = ["X", ".", "X"]
print(row[0] == "x", row[0] == "X", row[1] == ".")
print("số X (đúng):", row.count("X"), "| số x (sai chữ hoa/thường):", row.count("x"))

False True True
số X (đúng): 2 | số x (sai chữ hoa/thường): 0


## 14. Mẹo & nhận dạng pattern

- **"Đếm số thành phần liên thông trên lưới"** ⇒ DFS/BFS/Union-Find. Đây là khung chuẩn (như Number of Islands, LeetCode 200).
- **"Các hình có cấu trúc đặc biệt (đoạn thẳng, không dính nhau)"** ⇒ hãy tìm **đại diện duy nhất** (ô đầu tiên theo một thứ tự quét), rồi chỉ đếm đại diện.
- **"Rừng: `số cây = V - E`"** là mẹo hay cho đồ thị không chu trình; bạn gặp lại ở nhiều bài về đồ thị.
- **Follow-up "một lượt, `O(1)` bộ nhớ, không sửa input"** thường gợi ý một thuật toán **cục bộ**: quyết định chỉ nhìn các ô đã duyệt (trên/trái).
- **Thứ tự duyệt quyết định cái gì "đã biết"**: quét hàng-trước thì ô trên và trái đã thăm — dùng chúng làm "bộ nhớ" miễn phí.
- Luôn **kiểm tra biên** trước khi chỉ số `-1` (Python không báo lỗi!).

## 15. Biến thể

1. **Đếm đảo (LeetCode 200 Number of Islands):** hình tùy ý ⇒ chỉ DFS/BFS/Union-Find đúng; ô góc không đúng.
2. **Cho phép tàu dính nhau:** như trên, cần flood fill.
3. **Tìm độ dài tàu dài nhất / thống kê số tàu theo kích cỡ:** dùng DFS để đo `size` của mỗi thành phần, hoặc ở bảng hợp lệ, đếm độ dài run của `X` theo ô góc (đi sang phải/xuống dưới từ ô góc).
4. **Tàu đặt chéo cũng được (8 hướng):** đổi `dirs` thành 8 hướng trong DFS; mẹo ô góc không còn đúng.
5. **Bảng rất lớn không nhét vừa bộ nhớ (stream):** mẹo ô góc chỉ cần hàng trước đó ⇒ `O(n)` bộ nhớ khi đọc từng hàng — ý tưởng quét luồng.

In [34]:
# Biến thể 3: thống kê số tàu theo kích cỡ (với bảng hợp lệ), dùng ô góc rồi đo run
from collections import Counter

def ship_sizes(board):
    m, n = len(board), len(board[0])
    sizes = Counter()
    for i in range(m):
        for j in range(n):
            if board[i][j] != "X":
                continue
            if (i > 0 and board[i - 1][j] == "X") or (j > 0 and board[i][j - 1] == "X"):
                continue
            k = 1
            if j + 1 < n and board[i][j + 1] == "X":          # tàu ngang
                while j + k < n and board[i][j + k] == "X":
                    k += 1
            else:                                              # tàu dọc (hoặc 1x1)
                while i + k < m and board[i + k][j] == "X":
                    k += 1
            sizes[k] += 1
    return dict(sorted(sizes.items()))


print(ship_sizes(trace_board))                # {1: 2, 2: 2, 3: 1}
rng = random.Random(15)
for _ in range(300):
    board, truth, sizes = gen_board(rng.randint(1, 9), rng.randint(1, 9), 25, rng)
    assert ship_sizes(board) == dict(sorted(Counter(sizes).items()))
print("biến thể 3 khớp dữ liệu sinh")

{1: 2, 2: 2, 3: 1}
biến thể 3 khớp dữ liệu sinh


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS + visited | Tô loang từng tàu | `O(mn)` | `O(mn)` | Bảng bất kỳ; dễ nghĩ, quen thuộc |
| 2. Union-Find | Gộp ô `X` liền kề | `O(mn·α)` | `O(mn)` | Khi cần gộp động / đồ thị lớn |
| 3. `V - E` | Rừng: số cây = đỉnh - cạnh | `O(mn)` | `O(1)` | Bảng hợp lệ (không chu trình); ngắn gọn |
| 4. Ô góc trên-trái | Mỗi tàu một đại diện | `O(mn)` | `O(1)` | **Khuyên dùng**: đúng follow-up, không sửa bảng |

Cả bốn cho cùng kết quả trên bảng hợp lệ (mục 12); chỉ Cách 1 và 2 còn đúng cho bảng không hợp lệ (mục 13).

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `[["X", "."], [".", "X"]][-1][0]`; (b) `[["X", "."], [".", "X"]][0 - 1][1]`; (c) `sum(row.count("X") for row in ["XX.", ".X."])` nếu mỗi chuỗi coi như một hàng; (d) số ô góc của bảng `["XX", ".."]`.

**Bài 2 (tính tay).** Cho bảng
```
X . X X
X . . .
. . X .
```
Liệt kê các ô góc và cho biết có bao nhiêu tàu. Sau đó tính lại bằng `V - E`.

**Bài 3 (điền TODO).** Hoàn thành `count_todo(board)` theo Cách 4.

In [35]:
def count_todo(board):
    # TODO 1: lấy m, n = len(board), len(board[0]) và ans = 0
    # TODO 2: hai vòng for i, j; bỏ qua ô '.'
    # TODO 3: bỏ qua nếu i > 0 và board[i-1][j] == 'X'
    # TODO 4: bỏ qua nếu j > 0 và board[i][j-1] == 'X'
    # TODO 5: còn lại thì ans += 1; cuối cùng trả ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `count_bfs(board)` dùng `collections.deque` (BFS) thay vì ngăn xếp, **không nhìn lại** Cách 1. Kiểm tra với ví dụ đề.

**Bài 5 (biến thể / mini challenge).** Viết `largest_ship(board)` trả về **độ dài tàu lớn nhất** (số ô). Kiểm tra: bảng ở Bài 2 → `2`; ví dụ 1 của đề → `3`; bảng `[["."]]` → `0`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: chỉ số `-1` là hàng/cột cuối. `[-1][0]` ở bảng 2x2 là hàng cuối cột 0 = `'.'`. Với (d), `(0,0)` là góc, `(0,1)` có `X` bên trái nên không.
- **Mức 2 (vừa):** Bài 2: ô góc `(0,0)`, `(0,2)`, `(2,2)` ⇒ 3 tàu. `V = 5`, `E = 2` (`(1,0)` có `X` trên; `(0,3)` có `X` trái) ⇒ `5 - 2 = 3`. Bài 4: giống Cách 1 nhưng `queue.popleft()` thay cho `stack.pop()`.
- **Mức 3 (sâu):** Bài 5: dùng DFS/BFS để đo `size` mỗi thành phần và lấy `max`; hoặc cách bảng hợp lệ: ở mỗi ô góc, đếm run sang phải rồi xuống dưới (như biến thể 3).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [36]:
from collections import deque

# Bài 1
b2 = [["X", "."], [".", "X"]]
assert b2[-1][0] == "."
assert b2[0 - 1][1] == "X"
assert sum(row.count("X") for row in ["XX.", ".X."]) == 3
assert count_corner([list("XX"), list("..")]) == 1

# Bài 2
ex2 = [list("X.XX"), list("X..."), list("..X.")]
corners = [(i, j) for i in range(3) for j in range(4)
           if ex2[i][j] == "X" and (i == 0 or ex2[i - 1][j] != "X") and (j == 0 or ex2[i][j - 1] != "X")]
assert corners == [(0, 0), (0, 2), (2, 2)]
assert count_forest(ex2) == 3 == count_dfs(ex2)


# Bài 3
def count_todo_ans(board):
    m, n = len(board), len(board[0])
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == ".":
                continue
            if i > 0 and board[i - 1][j] == "X":
                continue
            if j > 0 and board[i][j - 1] == "X":
                continue
            ans += 1
    return ans


for board, expected in examples:
    assert count_todo_ans(board) == expected


# Bài 4
def count_bfs(board):
    m, n = len(board), len(board[0])
    visited = [[False] * n for _ in range(m)]
    ans = 0
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X" and not visited[i][j]:
                ans += 1
                visited[i][j] = True
                q = deque([(i, j)])
                while q:
                    x, y = q.popleft()
                    for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "X" and not visited[nx][ny]:
                            visited[nx][ny] = True
                            q.append((nx, ny))
    return ans


for board, expected in examples:
    assert count_bfs(board) == expected
assert count_bfs(ex2) == 3


# Bài 5
def largest_ship(board):
    m, n = len(board), len(board[0])
    best = 0
    seen = set()
    for i in range(m):
        for j in range(n):
            if board[i][j] == "X" and (i, j) not in seen:
                seen.add((i, j))
                stack, size = [(i, j)], 0
                while stack:
                    x, y = stack.pop()
                    size += 1
                    for dx, dy in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and board[nx][ny] == "X" and (nx, ny) not in seen:
                            seen.add((nx, ny))
                            stack.append((nx, ny))
                best = max(best, size)
    return best


assert largest_ship(ex2) == 2
assert largest_ship(examples[0][0]) == 3
assert largest_ship([["."]]) == 0
rng = random.Random(19)
for _ in range(300):
    board, truth, sizes = gen_board(rng.randint(1, 9), rng.randint(1, 9), 25, rng)
    assert largest_ship(board) == (max(sizes) if sizes else 0)
    assert count_bfs(board) == truth
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (đếm ô góc trên-trái, `O(1)` bộ nhớ)

```python
class Solution:
    def countBattleships(self, board: List[List[str]]) -> int:
        m, n = len(board), len(board[0])
        ans = 0
        for i in range(m):
            for j in range(n):
                if board[i][j] == ".":
                    continue
                if i > 0 and board[i - 1][j] == "X":
                    continue
                if j > 0 and board[i][j - 1] == "X":
                    continue
                ans += 1
        return ans
```

**Checklist**

- [ ] Đã kiểm tra biên `i > 0` và `j > 0` trước khi nhìn `board[i-1][j]`, `board[i][j-1]` (chỉ số âm **không** báo lỗi)?
- [ ] Đã so sánh với `'X'` (chữ hoa) và `'.'`?
- [ ] Đã nhớ mẹo ô góc **chỉ** đúng với bảng hợp lệ (tàu thẳng, không dính nhau)?
- [ ] Nếu dùng DFS: `visited` dùng chung cho toàn bộ vòng ngoài, đánh dấu **khi đẩy vào ngăn xếp**?
- [ ] Không sửa `board` (follow-up)?
- [ ] Đã test: `1x1`, một hàng, một cột, chạm chéo, toàn trống, và so với DFS trên bảng ngẫu nhiên?